# Study 9 (PREREG): per-condition 4-D inference — independent fit for each condition

Companion to `model-study9-vbmc-laplace-beta.ipynb`, but instead of one GP shared across all
conditions, we fit a **separate** 4-D posterior `(length_scale, mu_0, sigma, beta_speaker)`
*independently for each condition* (diet / personality / physical / heterogeneous). The point
is to see whether the fitted **length scale** differs by condition — i.e. whether some feature
domains generalize over a wider region of embedding space than others.

Mechanics: `inference.run_vbmc_free_shapes_beta` iterates over whatever groups are in the
`responses_cond` dict it is handed (fitting the profiled linking shapes to just those
participants and summing log Z over just those groups). Passing a **single-condition** dict
`{c: responses_cond[c]}` therefore fits that condition alone — same objective, same priors, no
pooling across conditions.

Only the **fitted (profiled) linking shapes** variant is run here (the one used downstream in
the pooled notebook). Data is the **preregistered** sample `data/study9_prereg.csv`.

Results cached under `results/study9-vbmc-laplace-beta-percond/posterior-<condition>.npz` —
delete a file to refit that condition.

In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("JAX_ENABLE_X64", "1")

import sys
sys.path.insert(0, ".")

import json
import numpy as np
import matplotlib.pyplot as plt

import inference as inf   # Laplace + VBMC machinery (sets jax x64; import before other jax users)

MAX_EVALS = 400
DATA_CSV    = '../../data/study9_prereg.csv'          # PREREGISTERED sample
RESULTS_DIR = os.path.join('results', 'study9-vbmc-laplace-beta-percond')
os.makedirs(RESULTS_DIR, exist_ok=True)

geom = inf.load_geometry()
J = geom['J']
test_feature_names, test_trait = geom['test_feature_names'], geom['test_trait']
responses_cond, _ = inf.load_responses(DATA_CSV)
emp_cond = {c: np.array(responses_cond[c]).mean(0) for c in inf.CONDITIONS}
N_cond   = {c: int(responses_cond[c].shape[0]) for c in inf.CONDITIONS}
for c in inf.CONDITIONS:
    print(f"  {c:<14}: {N_cond[c]} participants  mean rating={np.array(responses_cond[c]).mean():.3f}")

  diet          : 98 participants  mean rating=0.537
  personality   : 103 participants  mean rating=0.533
  physical      : 99 participants  mean rating=0.597
  heterogeneous : 101 participants  mean rating=0.501


## 1. Fit each condition independently (4-D, fitted shapes)

For each condition we run `run_vbmc_free_shapes_beta` on a one-condition `responses_cond`
dict. Each fit is cached to its own `posterior-<condition>.npz` (same keys as the pooled
notebook: `ls/mu0/sigma/beta_samples` + meta).

In [ ]:
SAMPLE_KEYS = ('ls_samples', 'mu0_samples', 'sigma_samples', 'beta_samples')
META_KEYS   = ('elbo', 'func_count', 'convergence_status', 'runtime_s')

def load_or_fit(path, fit_fn):
    if os.path.exists(path):
        z = np.load(path, allow_pickle=True)
        fit = {k: z[k] for k in SAMPLE_KEYS}
        fit.update(json.loads(str(z['meta'])))
        print(f"loaded cached posterior from {path}  ({fit['convergence_status']})")
    else:
        fit = fit_fn()
        np.savez_compressed(path, **{k: fit[k] for k in SAMPLE_KEYS},
                            meta=json.dumps({k: fit[k] for k in META_KEYS}))
        print(f"saved posterior to {path}")
    return fit

fits = {}
for c in inf.CONDITIONS:
    print(f"\n=== condition: {c} (N={N_cond[c]}) ===")
    path = os.path.join(RESULTS_DIR, f'posterior-{c}.npz')
    fits[c] = load_or_fit(
        path,
        lambda c=c: inf.run_vbmc_free_shapes_beta(geom, {c: responses_cond[c]}, max_evals=MAX_EVALS))


=== condition: diet (N=98) ===


/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


Reshaping x0 to row vector.
Reshaping lower bounds to (1, 4).
Reshaping upper bounds to (1, 4).
Reshaping plausible lower bounds to (1, 4).
Reshaping plausible upper bounds to (1, 4).
Beginning variational optimization assuming NOISY observations of the log-joint
 Iteration  f-count    Mean[ELBO]    Std[ELBO]    sKL-iter[q]   K[q]  Convergence  Action
  eval   1: ls=0.300  mu_0=+0.000  sigma=1.500  beta=3.000  log_lik=276.5  log_joint=276.4
  eval   2: ls=1.718  mu_0=-1.695  sigma=1.622  beta=3.492  log_lik=0.4  log_joint=-1.4
  eval   3: ls=2.227  mu_0=+1.000  sigma=0.831  beta=5.362  log_lik=192.2  log_joint=190.9
  eval   4: ls=1.280  mu_0=+0.923  sigma=0.816  beta=7.650  log_lik=163.1  log_joint=161.8
  eval   5: ls=0.701  mu_0=-1.386  sigma=0.604  beta=1.261  log_lik=267.1  log_joint=265.3
  eval   6: ls=2.473  mu_0=-1.330  sigma=2.029  beta=6.912  log_lik=206.3  log_joint=204.4
  eval   7: ls=0.626  mu_0=+0.551  sigma=0.474  beta=2.198  log_lik=313.7  log_joint=312.8
  eval   8: 

/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/cma/evolution_strategy.py:3379: RuntimeWarning: invalid value encountered in scalar subtract
  current_fitness_range < opts['tolfunrel'] * (es.fit.median0 - es.fit.median_min),
/opt/miniconda3/envs/compgenerics/lib/python3.11/site-packages/cma/evolution_strategy.py:3379: RuntimeWarning: invalid value encountered in scalar multiply
  current_fitness_range < opts['tolfunrel'] * (es.fit.median0 - es.fit.median_min),


  eval  34: ls=3.997  mu_0=-0.097  sigma=0.377  beta=1.009  log_lik=307.1  log_joint=304.6
  eval  35: ls=0.272  mu_0=-0.161  sigma=0.390  beta=0.269  log_lik=296.7  log_joint=292.8
     5         35         310.45         6.15        19.29        2        329     
  eval  36: ls=3.655  mu_0=-0.214  sigma=0.299  beta=0.727  log_lik=299.6  log_joint=296.4
  eval  37: ls=0.059  mu_0=-0.052  sigma=0.279  beta=0.295  log_lik=299.4  log_joint=294.3
  eval  38: ls=3.331  mu_0=+0.033  sigma=0.279  beta=3.607  log_lik=312.0  log_joint=309.8
  eval  39: ls=0.783  mu_0=-0.107  sigma=0.555  beta=2.175  log_lik=311.1  log_joint=310.5
  eval  40: ls=0.873  mu_0=-0.315  sigma=0.065  beta=2.359  log_lik=283.8  log_joint=278.7
     6         40         307.95         8.01       392.71        2   6.56e+03     
  eval  41: ls=3.386  mu_0=-0.101  sigma=0.236  beta=2.032  log_lik=306.7  log_joint=304.1
  eval  42: ls=0.092  mu_0=-0.242  sigma=0.290  beta=2.054  log_lik=291.8  log_joint=289.7
  eval  43: l

## 2. Fitted parameters per condition

Posterior median [2.5, 97.5] for each parameter, one row per condition. The **length_scale**
column is the comparison of interest.

In [ ]:
def fmt(s): return f"{np.median(s):7.3f} [{np.percentile(s,2.5):.3f}, {np.percentile(s,97.5):.3f}]"

print(f"{'condition':<14}{'convergence':>16}{'evals':>7}{'ELBO':>9}")
for c in inf.CONDITIONS:
    f = fits[c]
    print(f"{c:<14}{f['convergence_status']:>16}{f['func_count']:>7}{f['elbo']:>9.1f}")

print()
print(f"{'condition':<14}{'length_scale':>26}{'mu_0':>26}{'sigma':>26}{'beta_speaker':>26}")
for c in inf.CONDITIONS:
    f = fits[c]
    print(f"{c:<14}{fmt(f['ls_samples']):>26}{fmt(f['mu0_samples']):>26}"
          f"{fmt(f['sigma_samples']):>26}{fmt(f['beta_samples']):>26}")

## 3. Length-scale (and other parameter) posteriors, overlaid by condition

Each condition's posterior marginal on the same axes. If the length-scale posteriors separate,
the domains generalize over meaningfully different spatial extents; if they overlap, a single
shared length scale (the pooled fit) is justified.

In [ ]:
ccolor = {'diet': 'tab:green', 'personality': 'tab:blue',
          'physical': 'tab:red', 'heterogeneous': 'tab:gray'}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
param_specs = [('ls_samples', 'length_scale'), ('mu0_samples', 'mu_0'),
               ('sigma_samples', 'output_scale (sigma)'), ('beta_samples', 'beta_speaker')]
for ax, (key, name) in zip(axes, param_specs):
    for c in inf.CONDITIONS:
        ax.hist(fits[c][key], bins=50, density=True, histtype='step', lw=2,
                color=ccolor[c], label=c)
        ax.axvline(np.median(fits[c][key]), color=ccolor[c], ls=':', alpha=0.7)
    if key == 'beta_samples':
        ax.axvline(inf.BETA_SPEAKER, color='k', ls='--', lw=1, label='design value 3')
    ax.set_xlabel(name); ax.set_title(f'Posterior: {name}')
axes[0].legend(fontsize=8)
fig.suptitle('Per-condition 4-D posteriors (fitted shapes, prereg data)', fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
out_png = os.path.join(RESULTS_DIR, 'per-condition-marginals.png')
fig.savefig(out_png, dpi=150, bbox_inches='tight')
print('saved', out_png)
plt.show()

### Length-scale comparison (forest plot)

Just the length scale, as median with 95% credible interval per condition — the cleanest read
on whether the fitted spatial extent differs by domain.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, c in enumerate(inf.CONDITIONS):
    s = fits[c]['ls_samples']
    med = np.median(s)
    lo, hi = np.percentile(s, [2.5, 97.5])
    ax.errorbar(med, i, xerr=[[med - lo], [hi - med]], fmt='o', ms=8, capsize=4,
                color=ccolor[c], lw=2)
    ax.text(hi, i, f'  {med:.3f} [{lo:.3f}, {hi:.3f}]', va='center', fontsize=8)
ax.set_yticks(range(len(inf.CONDITIONS)))
ax.set_yticklabels(inf.CONDITIONS)
ax.set_ylim(-0.6, len(inf.CONDITIONS) - 0.4)
ax.invert_yaxis()
ax.set_xlabel('length_scale (posterior median, 95% CI)')
ax.set_title('Fitted GP length scale by condition (independent fits)')
plt.tight_layout()
out_png = os.path.join(RESULTS_DIR, 'length-scale-forest.png')
fig.savefig(out_png, dpi=150, bbox_inches='tight')
print('saved', out_png)
plt.show()

## 4. GP coherence field per condition at each condition's own posterior median

Same visualization as the pooled notebook's section 6 and `plot-gp-coherence-field.ipynb`, but
here each panel's contours use **that condition's own** fitted `(ls, mu_0, sigma, beta)` — so
the field reflects the independently fitted length scale. Test points are colored by empirical
mean rating on the same scale.

In [ ]:
all_x = np.vstack([np.array(geom['x_test'])] + [np.array(geom['x_train_cond'][c]) for c in inf.CONDITIONS])
pad = 0.08
(x_lo, y_lo), (x_hi, y_hi) = all_x.min(0) - pad, all_x.max(0) + pad
gx, gy = np.meshgrid(np.linspace(x_lo, x_hi, 80), np.linspace(y_lo, y_hi, 80))
X_grid = np.column_stack([gx.ravel(), gy.ravel()])
x_test_np = np.array(geom['x_test'])

# per-condition posterior-median theta
theta_med = {c: (float(np.median(fits[c]['ls_samples'])), float(np.median(fits[c]['mu0_samples'])),
                 float(np.median(fits[c]['sigma_samples'])), float(np.median(fits[c]['beta_samples'])))
             for c in inf.CONDITIONS}

fig, axes = plt.subplots(2, 2, figsize=(13.5, 12))
for ax, c in zip(axes.ravel(), inf.CONDITIONS):
    LS_c, MU_c, SIG_c, BETA_c = theta_med[c]
    field    = inf.gp_coherence_field(geom, c, LS_c, MU_c, SIG_c, X_grid,
                                      beta_speaker=BETA_c).reshape(gx.shape)
    pz1_test = inf.gp_coherence_field(geom, c, LS_c, MU_c, SIG_c, x_test_np, beta_speaker=BETA_c)
    cs = ax.contourf(gx, gy, field, levels=20, cmap='Blues')
    cb = fig.colorbar(cs, ax=ax); cb.set_label('GP P(kind-linked)')

    x_tr = np.array(geom['x_train_cond'][c])
    ax.scatter(x_tr[:, 0], x_tr[:, 1], s=60, color='#16305e', edgecolor='k',
               zorder=3, label='train (kind-linked)')
    for (xx, yy), nm in zip(x_tr, geom['train_names_cond'][c]):
        ax.annotate(nm[:12], (xx, yy), fontsize=4, xytext=(2, 3), textcoords='offset points')
    ax.scatter(x_test_np[:, 0], x_test_np[:, 1], s=80, c=emp_cond[c], cmap='Blues', norm=cs.norm,
               edgecolor='k', linewidth=1.2, zorder=3, label='test (mean response)')

    r = np.corrcoef(pz1_test, emp_cond[c])[0, 1]
    ax.set_title(f"{c.capitalize()}  (n={N_cond[c]})\n"
                 f"ls={LS_c:.3f}, beta={BETA_c:.2f}   r(field @ test, mean) = {r:+.2f}")
    ax.set_xlabel('Embedding Dim 1'); ax.set_ylabel('Embedding Dim 2')
    ax.legend(fontsize=7, loc='upper right')

fig.suptitle("Per-condition coherence fields at each condition's own posterior median\n"
             "(independent 4-D fits, fitted shapes, prereg data)", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.965])
out_png = os.path.join(RESULTS_DIR, 'gp-coherence-field-percond.png')
fig.savefig(out_png, dpi=150, bbox_inches='tight')
print('saved', out_png)
plt.show()